# Knowledge Distillation from Large Teacher Models

Training large language models is expensive; running them at inference time is also expensive. Knowledge distillation transfers the learned behavior of a large **teacher** model into a smaller, faster **student** model — without requiring the student to be trained from scratch on raw data. The core insight is that a teacher's output distribution over the vocabulary carries far more information than a one-hot label: a soft probability vector encodes which tokens are similar, which are plausible alternatives, and where the teacher is uncertain. A student trained on these soft targets learns more efficiently than one trained on hard labels alone.

## Soft Targets

Standard supervised training minimizes the cross-entropy between the model's predictions and one-hot targets:

$$\mathcal{L}_{\text{CE}} = -\sum_v \mathbf{1}[v = y^*] \log p_\theta(v).$$

Knowledge distillation replaces the one-hot target with the teacher's output distribution $q_T$ — the **soft target**. Hinton et al. [@hinton2015distilling] introduced a temperature parameter $\tau$ to soften the teacher distribution further:

$$q_T^\tau(v) = \frac{\exp(z_v^T / \tau)}{\sum_{v'} \exp(z_{v'}^T / \tau)},$$

where $z^T$ are the teacher's logits. At $\tau = 1$ the distribution is unchanged; at $\tau > 1$ the distribution is flatter, emphasizing the relative similarities between tokens; at $\tau \to \infty$ the distribution approaches uniform.

The **distillation loss** is the KL divergence from the student to the (temperature-scaled) teacher:

$$\mathcal{L}_{\text{KD}} = \tau^2 \cdot D_{\text{KL}}\!\left(q_T^\tau \,\|\, q_S^\tau\right) = \tau^2 \sum_v q_T^\tau(v) \log \frac{q_T^\tau(v)}{q_S^\tau(v)}.$$

The $\tau^2$ factor re-scales the gradients to account for the division by $\tau$ inside the softmax — without it, gradient magnitudes would decrease at large $\tau$, requiring $\tau$-dependent learning rate tuning.

:::{.callout-note}
The $\tau^2$ rescaling is essential for maintaining consistent gradient magnitudes across temperature values. It ensures the effective learning rate does not need to be retuned when changing $\tau$.

:::

## The Combined Objective

In practice, distillation is combined with the standard cross-entropy loss on hard labels. This ensures the student still performs well on the ground-truth task:

$$\mathcal{L} = (1 - \alpha) \cdot \mathcal{L}_{\text{CE}} + \alpha \cdot \mathcal{L}_{\text{KD}},$$

where $\alpha \in [0, 1]$ balances the two losses. Setting $\alpha = 1$ trains entirely on teacher knowledge; $\alpha = 0$ ignores the teacher. Typical values: $\alpha = 0.5$–$0.9$, $\tau = 2$–$4$.

Implementing the combined distillation loss:

In [ ]:
import torch
import torch.nn.functional as F


def distillation_loss(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    labels: torch.Tensor,
    temperature: float = 2.0,
    alpha: float = 0.7,
) -> tuple[torch.Tensor, dict]:
    """Combined distillation + cross-entropy loss (Hinton et al., 2015).

    Args:
        student_logits: shape (B, T, V) — student model logits.
        teacher_logits: shape (B, T, V) — teacher model logits (no grad needed).
        labels: shape (B, T) — ground-truth token ids; -100 for masked positions.
        temperature: softening temperature tau (>= 1).
        alpha: weight on KL distillation loss; (1 - alpha) weight on CE.

    Returns:
        loss: scalar tensor (differentiable w.r.t. student_logits).
        metrics: dict with 'ce_loss' and 'kd_loss' for monitoring.
    """
    # Reshape for loss computation
    B, T, V = student_logits.shape
    student_logits_flat = student_logits.view(-1, V)    # (B*T, V)
    teacher_logits_flat = teacher_logits.view(-1, V)    # (B*T, V)
    labels_flat = labels.view(-1)                       # (B*T,)

    # Hard-label cross-entropy
    ce_loss = F.cross_entropy(student_logits_flat, labels_flat, ignore_index=-100)  # <1>

    # Soft-target KL divergence
    mask = labels_flat != -100                          # <2>
    if mask.sum() == 0:
        return ce_loss, {"ce_loss": ce_loss.item(), "kd_loss": 0.0}

    s_log_soft = F.log_softmax(student_logits_flat[mask] / temperature, dim=-1)  # <3>
    t_soft = F.softmax(teacher_logits_flat[mask] / temperature, dim=-1)

    kd_loss = F.kl_div(
        s_log_soft,
        t_soft,
        reduction="batchmean",
    ) * (temperature ** 2)                              # <4>

    loss = (1 - alpha) * ce_loss + alpha * kd_loss
    return loss, {"ce_loss": ce_loss.item(), "kd_loss": kd_loss.item()}

1. Standard next-token prediction loss on hard labels; `-100` positions are ignored.
2. Mask out padding/prompt positions — we only compute the KL where the label is valid.
3. Apply temperature scaling to both student and teacher before softmax/log-softmax.
4. The $\tau^2$ factor rescales gradients to match $\tau = 1$ magnitude.

## Forward KL vs Reverse KL

The choice of KL direction matters. The standard distillation loss uses **forward KL** $D_{\text{KL}}(q_T \| q_S)$, which is [mean-seeking]{.mark}: the student is penalized for assigning low probability anywhere the teacher assigns high probability. The student is pushed to cover all of the teacher's modes.

An alternative is **reverse KL** $D_{\text{KL}}(q_S \| q_T)$, which is [mode-seeking]{.mark}: the student is penalized for assigning high probability where the teacher assigns low probability. The student concentrates on the teacher's dominant modes and ignores the tails.

For language models:

- **Forward KL** produces more conservative outputs — the student tries to assign non-negligible probability to everything the teacher considers plausible. This is generally preferred for distillation.
- **Reverse KL** produces more confident, peaked outputs — the student picks one good mode and ignores alternatives. This can cause overconfidence and repetition.

The forward KL is the standard choice. Implementing both for comparison:

In [ ]:
def forward_kl(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    temperature: float = 1.0,
) -> torch.Tensor:
    """Forward KL: KL(teacher || student). Mean-seeking.

    Args:
        student_logits: shape (N, V).
        teacher_logits: shape (N, V).
        temperature: softening temperature.

    Returns:
        Scalar KL divergence (batchmean reduction).
    """
    s_log = F.log_softmax(student_logits / temperature, dim=-1)
    t = F.softmax(teacher_logits / temperature, dim=-1)
    return F.kl_div(s_log, t, reduction="batchmean") * (temperature ** 2)


def reverse_kl(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    temperature: float = 1.0,
) -> torch.Tensor:
    """Reverse KL: KL(student || teacher). Mode-seeking.

    Args:
        student_logits: shape (N, V).
        teacher_logits: shape (N, V).
        temperature: softening temperature.

    Returns:
        Scalar KL divergence (batchmean reduction).
    """
    s = F.softmax(student_logits / temperature, dim=-1)
    s_log = F.log_softmax(student_logits / temperature, dim=-1)
    t_log = F.log_softmax(teacher_logits / temperature, dim=-1)
    return (s * (s_log - t_log)).sum(dim=-1).mean() * (temperature ** 2)

## Sequence-Level Distillation

Token-level distillation (as above) trains on teacher logits at every position in the training sequence. A complementary approach is **sequence-level distillation** [@kim2016sequencelevel]: instead of matching logits, we use the teacher to generate a synthetic dataset and train the student on that data with standard cross-entropy.

The procedure:

1. Sample responses from the teacher: $y \sim p_T(\cdot \mid x)$.
2. Add $(x, y)$ pairs to the training set.
3. Train the student with cross-entropy on the augmented dataset.

Sequence-level distillation is simpler (no need for teacher logits at inference time) but wastes the rich probability information in the teacher's full distribution. It is particularly useful when only the teacher's outputs are available — for example, when distilling from a proprietary API model.

Implementing `generate_teacher_dataset`:

In [ ]:
from torch.utils.data import Dataset


def generate_teacher_dataset(
    teacher_model,
    tokenizer,
    prompts: list[str],
    n_samples_per_prompt: int = 1,
    max_new_tokens: int = 256,
    temperature: float = 1.0,
    device: torch.device = None,
) -> list[dict]:
    """Generate a synthetic dataset by sampling from the teacher model.

    Args:
        teacher_model: frozen teacher language model.
        tokenizer: tokenizer.
        prompts: list of prompt strings.
        n_samples_per_prompt: how many responses to generate per prompt.
        max_new_tokens: max generation length.
        temperature: sampling temperature.
        device: compute device.

    Returns:
        List of dicts with keys 'input_ids' and 'labels' (same as input_ids,
        shifted for next-token prediction).
    """
    if device is None:
        device = next(teacher_model.parameters()).device

    teacher_model.eval()
    records = []

    with torch.no_grad():
        for prompt in prompts:
            prompt_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

            for _ in range(n_samples_per_prompt):
                output_ids = teacher_model.generate(
                    prompt_ids,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    do_sample=(temperature > 0),
                )[0]
                records.append({
                    "input_ids": output_ids.cpu(),
                    "labels": output_ids.cpu().clone(),
                })

    return records

## The Distillation Training Loop

For token-level distillation the loop follows the same structure as SFT ([NB08](/courses/llm/08-sft-lora.html)), with two additions: (1) a frozen teacher forward pass to obtain teacher logits, and (2) the combined distillation loss instead of plain cross-entropy.

The teacher is always run in `torch.no_grad()` — its parameters are frozen and only the student is updated.

In [ ]:
from dataclasses import dataclass
from torch.optim import AdamW
from torch.utils.data import DataLoader


@dataclass
class DistillConfig:
    temperature: float = 2.0    # KL softening temperature
    alpha: float = 0.7          # weight on distillation loss
    lr: float = 2e-4
    batch_size: int = 8
    max_steps: int = 5000
    eval_every: int = 500
    output_dir: str = "distill_checkpoints"


def distill_train(
    student_model,
    teacher_model,
    train_dataset,
    cfg: DistillConfig,
) -> dict:
    """Token-level knowledge distillation training loop.

    Args:
        student_model: the smaller model being trained (parameters updated).
        teacher_model: the larger frozen model (only forward pass, no grad).
        train_dataset: dataset yielding dicts with 'input_ids' and 'labels'.
        cfg: DistillConfig.

    Returns:
        history: dict with 'step', 'loss', 'ce_loss', 'kd_loss'.
    """
    import os

    device = next(student_model.parameters()).device

    # Freeze teacher
    for p in teacher_model.parameters():
        p.requires_grad_(False)
    teacher_model.eval()

    optimizer = AdamW(student_model.parameters(), lr=cfg.lr)
    loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)

    history = {"step": [], "loss": [], "ce_loss": [], "kd_loss": []}
    step = 0

    student_model.train()
    while step < cfg.max_steps:
        for batch in loader:
            if step >= cfg.max_steps:
                break

            input_ids = batch["input_ids"].to(device)      # (B, T)
            labels = batch["labels"].to(device)            # (B, T)

            # Student forward (with grad)
            student_logits = student_model(input_ids)       # (B, T, V)

            # Teacher forward (no grad)
            with torch.no_grad():
                teacher_logits = teacher_model(input_ids)   # (B, T, V)

            # Shift labels for next-token prediction
            shift_logits_s = student_logits[:, :-1, :].contiguous()
            shift_logits_t = teacher_logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss, m = distillation_loss(
                shift_logits_s,
                shift_logits_t,
                shift_labels,
                temperature=cfg.temperature,
                alpha=cfg.alpha,
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            step += 1

            if step % cfg.eval_every == 0:
                history["step"].append(step)
                history["loss"].append(loss.item())
                history["ce_loss"].append(m["ce_loss"])
                history["kd_loss"].append(m["kd_loss"])
                print(
                    f"step {step:5d} | loss {loss.item():.4f} "
                    f"| ce {m['ce_loss']:.4f} | kd {m['kd_loss']:.4f}"
                )

    os.makedirs(cfg.output_dir, exist_ok=True)
    torch.save(student_model.state_dict(),
               os.path.join(cfg.output_dir, "student_final.pt"))
    return history

## Layer-Matching Distillation

Token-level output distillation matches only the final-layer distributions. **Layer-matching distillation** (also called feature-level or intermediate distillation) additionally aligns the student's intermediate representations — hidden states, attention maps, or FFN outputs — with those of the teacher.

The idea: each student layer $l$ is matched to a corresponding teacher layer $f(l)$ via a mean-squared error objective:

$$\mathcal{L}_{\text{feat}} = \sum_{l} \left\| W_l \cdot H_l^S - H_{f(l)}^T \right\|_2^2,$$

where $H_l^S$ and $H_{f(l)}^T$ are the hidden states at the matched layers, and $W_l$ is a learned linear projection to align dimensions (since the student is typically narrower than the teacher).

Implementing the feature-alignment projection and loss:

In [ ]:
import torch.nn as nn


class FeatureAlignmentProjection(nn.Module):
    """Linear projection from student hidden dim to teacher hidden dim.

    Used to align intermediate representations for layer-matching distillation.

    Args:
        student_dim: hidden dimension of the student model.
        teacher_dim: hidden dimension of the teacher model.
    """

    def __init__(self, student_dim: int, teacher_dim: int) -> None:
        super().__init__()
        self.proj = nn.Linear(student_dim, teacher_dim, bias=False)

    def forward(self, student_hidden: torch.Tensor) -> torch.Tensor:
        """Project student hidden states to teacher dimension.

        Args:
            student_hidden: shape (B, T, student_dim).

        Returns:
            projected: shape (B, T, teacher_dim).
        """
        return self.proj(student_hidden)


def feature_distillation_loss(
    student_hiddens: list[torch.Tensor],
    teacher_hiddens: list[torch.Tensor],
    projections: nn.ModuleList,
) -> torch.Tensor:
    """MSE loss between projected student and teacher hidden states.

    Args:
        student_hiddens: list of (B, T, d_s) tensors, one per matched student layer.
        teacher_hiddens: list of (B, T, d_t) tensors, one per matched teacher layer.
        projections: ModuleList of FeatureAlignmentProjection modules.

    Returns:
        Scalar MSE loss averaged over layers and tokens.
    """
    assert len(student_hiddens) == len(teacher_hiddens) == len(projections)
    total = torch.tensor(0.0, device=student_hiddens[0].device)
    for h_s, h_t, proj in zip(student_hiddens, teacher_hiddens, projections):
        projected = proj(h_s)                            # (B, T, d_t)
        total = total + F.mse_loss(projected, h_t.detach())
    return total / len(student_hiddens)

:::{.callout-note}
Layer-matching distillation requires accessing intermediate hidden states. In practice this means registering forward hooks on both the teacher and student, or modifying the model's `forward()` to return a list of hidden states alongside logits. The benefit is substantially better transfer of structural knowledge — attention patterns and feature hierarchies — beyond what the output distribution alone captures.

:::

## Distillation for Post-Training

Distillation is particularly important in the post-training phase for language models. The most common use case: after training a large reasoning model with GRPO ([NB10](/courses/llm/10-grpo.html)), distill its reasoning behavior into a smaller student.

This is the approach used in DeepSeek-R1 [@deepseek2025]: a 671B MoE teacher generates chains of thought (CoTs) with correct reasoning traces, and these are used to fine-tune smaller dense student models (1.5B, 7B, 14B, 32B parameters) via standard SFT — a form of sequence-level distillation from the teacher's generations.

The recipe:

1. Sample many reasoning problems and generate teacher solutions (with intermediate reasoning steps).
2. Filter to keep only correct solutions.
3. SFT the student on `(problem, reasoning_trace, answer)` triples.

This is significantly cheaper than running full GRPO on the student — the student never needs to interact with a reward model. The cost is that the student's reasoning quality is bounded by the teacher's, and it inherits the teacher's failure modes.

In [ ]:
def filter_correct_solutions(
    problems: list[str],
    solutions: list[str],
    answers: list[str],
    verify_fn,
) -> list[dict]:
    """Filter teacher-generated solutions to keep only correct ones.

    Args:
        problems: list of problem strings.
        solutions: list of teacher-generated solution strings (may include CoT).
        answers: list of ground-truth answer strings.
        verify_fn: callable(solution, answer) -> bool.

    Returns:
        List of dicts with 'problem' and 'solution' keys,
        only for correct solutions.
    """
    correct = []
    for problem, solution, answer in zip(problems, solutions, answers):
        if verify_fn(solution, answer):
            correct.append({"problem": problem, "solution": solution})
    return correct

## Evaluating Distillation Quality

Three metrics characterize how well distillation succeeded:

- **Perplexity gap** — the ratio of student to teacher perplexity on a held-out corpus. A ratio close to 1.0 indicates the student has successfully absorbed the teacher's language model quality.
- **Task accuracy** — downstream benchmark performance. Perplexity can be low while task performance degrades; always measure both.
- **Compression ratio** — `teacher_params / student_params`. Typical targets: 4×–16× compression while retaining >90% of task performance.

Implementing a quick evaluation helper:

In [ ]:
import math
def evaluate_perplexity(
    model,
    dataset,
    batch_size: int = 4,
    device: torch.device = None,
) -> float:
    """Compute perplexity on a token dataset.

    Args:
        model: language model.
        dataset: dataset yielding dicts with 'input_ids' and 'labels'.
        batch_size: evaluation batch size.
        device: compute device.

    Returns:
        Perplexity (exp of mean cross-entropy loss).
    """
    if device is None:
        device = next(model.parameters()).device

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids)[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
                reduction="sum",
            )
            n_tokens = (shift_labels != -100).sum().item()
            total_loss += loss.item()
            total_tokens += n_tokens

    model.train()
    return math.exp(total_loss / max(total_tokens, 1))

## Summary

| Concept | Key detail |
|---|---|
| Soft targets | Teacher logits scaled by temperature $\tau$; dark knowledge in non-argmax probabilities. |
| $\tau^2$ rescaling | KL gradient scales as $1/\tau^2$; multiplying the KL term by $\tau^2$ keeps its magnitude stable as $\tau$ varies. |
| Combined loss | $\alpha \mathcal{L}_{\text{KD}} + (1 - \alpha) \mathcal{L}_{\text{CE}}$; $\alpha \approx 0.7$ is a common starting point. |
| Forward KL | Mode-covering — student spreads mass over all modes the teacher covers. Preferred for generation diversity. |
| Reverse KL | Mode-seeking — student concentrates on the teacher's highest-probability mode. Preferred for focused generation. |
| Sequence-level KD | Teacher generates full sequences; student trains on them as ground truth. Avoids exposure bias. |
| Feature alignment | Projecting student hidden states to match teacher hidden states layer-by-layer; MSE loss. |
| Post-training distillation | Distill a reasoning model's chain-of-thought behavior into a smaller model after GRPO. |

: {tbl-colwidths="[30,70]"}

## Exercises

1. **Temperature sweep.** Train a student with `temperature` $\in \{1, 2, 4, 8\}$ and fixed `alpha=0.7`. Plot final perplexity vs temperature. At what temperature does distillation quality peak?

2. **Alpha sweep.** Fix `temperature=2.0` and vary `alpha` $\in \{0, 0.3, 0.7, 1.0\}$. Compare student perplexity and downstream accuracy. What happens at `alpha=0` (standard SFT) vs `alpha=1` (pure distillation)?

3. **Forward vs reverse KL.** Replace `forward_kl` with `reverse_kl` in the distillation loss. Train both versions and compare: (a) output diversity, (b) perplexity, (c) tendency to produce repetitive text.

4. **Sequence-level distillation.** Use `generate_teacher_dataset` to create a synthetic training set from a teacher, then train the student on it with standard SFT. Compare against token-level distillation on the same compute budget.

5. **Layer matching.** Add `feature_distillation_loss` to the training loop, matching every other student layer to the corresponding teacher layer. Does intermediate alignment improve perplexity beyond output-only distillation?

6. **Reasoning distillation.** Simulate the DeepSeek-R1 recipe: generate CoT solutions from a large model on a math dataset (e.g. GSM8K), filter to correct solutions with `filter_correct_solutions`, and SFT a small model. Compare accuracy against the same small model trained without CoT data.

■